# 🌊 Tutorial: Differentiable Ocean Dynamics with Reactant.jl

## 1. What is this? (An Intuitive Introduction)
Imagine you are looking at a complex weather map. You see a storm, and you ask: **"What caused this?"** 

Usually, scientists have to run thousands of separate simulations, changing one thing at a time, to find the answer. This is slow and expensive.

**Automatic Differentiation** changes that. It allows us to run a single simulation and then "reverse the flow" to see exactly which factors (like wind or heat) influenced the outcome. 

- **The Forward Pass:** The ocean flows forward in time (Physics).
- **The Adjoint Pass:** The sensitivities flow backward in time (Cause-and-Effect).

By using **Reactant.jl** and **Enzyme.jl**, we make this process fast enough to run on powerful GPUs.


In [ ]:
using Pkg; Pkg.activate("."); Pkg.instantiate()
using Reactant, Enzyme, CUDA, MPI, Oceananigans
using Oceananigans.Architectures: ReactantState
include("ocean_utils.jl")

MPI.Init()
CUDA.versioninfo()


## 2. Setting the Scene: Our Ocean Domain
We divide the ocean into small cubes called **grid points**. To capture the most detail, we use smaller cubes near the surface where the wind and heat are strongest.


In [ ]:
const Nx = 48; const Ny = 96; const Nz = 32
const Lx = 1000.0 * 1e3; const Ly = 2000.0 * 1e3 

# Create a stretched vertical grid (higher resolution at surface)
k_center = collect(1:Nz)
Δz_center = @. 10 * 1.104^(Nz - k_center)
const Lz = sum(Δz_center)
z_faces = vcat([-Lz], -Lz .+ cumsum(Δz_center))
z_faces[Nz+1] = 0

# Physical parameters (simplified for this tutorial)
parameters = (Ly=Ly, Lz=Lz, ΔT=8, h=1000.0, y_sponge=1.9e6, λt=7days, 
              μ=1/30days, Lx=Lx, Nz=Nz)


## 3. Building the Ocean Architecture
We use  as our architecture. This simple command tells the simulation to run every calculation on the GPU using high-performance XLA kernels.


In [ ]:
# Define the architecture (GPU)
architecture = ReactantState()

# Build the Grid and the Mountainous Ocean Floor
grid = make_grid(architecture, Nx, Ny, Nz, Lx, Ly, Lz, z_faces, 4)

# Assemble the Ocean Model (Physics, Winds, Currents)
model = build_model(grid, 2.5minutes, parameters)

@info "Built "


## 4. Visualizing the Ocean State
After running the simulation, we can visualize the results. Red areas represent warmer water, while blue represents colder. The swirling whirlpools are **eddies**.


In [ ]:
using CairoMakie
# Visualizing the surface temperature
fig = Figure(size = (800, 400))
ax = Axis(fig[1, 1], title = "Surface Temperature")
heatmap!(ax, Array(interior(model.tracers.T, :, :, Nz)), colormap = :thermal)
fig
